In [1]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

In [2]:
class WIDERFaceYOLODataset(Dataset):
    def __init__(self, image_dir, label_dir, S=7, B=2, C=1, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.S = S
        self.B = B
        self.C = C

        self.image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
        self.image_files.sort()  # Ensure labels match images

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image_filename = self.image_files[index]
        image_path = os.path.join(self.image_dir, image_filename)
        label_path = os.path.join(self.label_dir, image_filename.replace('.jpg', '.txt').replace('.png', '.txt'))

        image = Image.open(image_path).convert("RGB")
        boxes = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    class_label, x_center, y_center, width, height = map(float, line.strip().split())
                    boxes.append([int(class_label), x_center, y_center, width, height])

        if self.transform:
            image = self.transform(image)

        label_matrix = torch.zeros((self.S, self.S, self.C + self.B * 5))

        for box in boxes:
            class_label, x, y, w, h = box
            i = min(int(self.S * y), self.S - 1)
            j = min(int(self.S * x), self.S - 1)
            x_cell, y_cell = self.S * x - j, self.S * y - i
            w_cell, h_cell = w * self.S, h * self.S


            if label_matrix[i, j, self.C] == 0:
                label_matrix[i, j, self.C] = 1  # object confidence
                label_matrix[i, j, self.C + 1:self.C + 5] = torch.tensor([x_cell, y_cell, w_cell, h_cell])
                label_matrix[i, j, 0] = class_label  # one-hot or class index

        return image, label_matrix


In [ ]:
import os
import random
from torch.utils.data import DataLoader, Subset

# Configuration
image_dir = 'WIDERFACE/images'
label_dir = 'WIDERFACE/labels'
S, B, C = 7, 2, 1
val_split = 0.2
batch_size = 16
seed = 42

# Transform (same for train/val unless you add augmentations later)
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

# Full dataset
full_dataset = WIDERFaceYOLODataset(image_dir=image_dir, label_dir=label_dir, S=S, B=B, C=C, transform=transform)

# Split indices
dataset_size = len(full_dataset)
indices = list(range(dataset_size))
random.seed(seed)
random.shuffle(indices)
split = int(val_split * dataset_size)
val_indices = indices[:split]
train_indices = indices[split:]

# Subset Datasets
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False)


In [4]:
print(len(train_dataset))
print(len(val_dataset))

10304
2576


In [5]:
import torch
import torch.nn as nn

class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=1):
        super(YOLOv1, self).__init__()
        self.S = S
        self.B = B
        self.C = C

        def conv_block(in_channels, out_channels, kernel_size, stride, padding):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.1)
            )

        # Feature Extractor
        self.features = nn.Sequential(
            conv_block(3, 64, 7, 2, 3),
            nn.MaxPool2d(2, 2),

            conv_block(64, 192, 3, 1, 1),
            nn.MaxPool2d(2, 2),

            conv_block(192, 128, 1, 1, 0),
            conv_block(128, 256, 3, 1, 1),
            conv_block(256, 256, 1, 1, 0),
            conv_block(256, 512, 3, 1, 1),
            nn.MaxPool2d(2, 2),

            # Repeat 1x1 + 3x3 blocks 4 times
            *[
                nn.Sequential(
                    conv_block(512, 256, 1, 1, 0),
                    conv_block(256, 512, 3, 1, 1)
                )
                for _ in range(4)
            ],

            conv_block(512, 512, 1, 1, 0),
            conv_block(512, 1024, 3, 1, 1),
            nn.MaxPool2d(2, 2),

            # Two times 1x1 + 3x3
            conv_block(1024, 512, 1, 1, 0),
            conv_block(512, 1024, 3, 1, 1),
            conv_block(1024, 512, 1, 1, 0),
            conv_block(512, 1024, 3, 1, 1),

            conv_block(1024, 1024, 3, 1, 1),
            conv_block(1024, 1024, 3, 2, 1),
            conv_block(1024, 1024, 3, 1, 1),
        )

        # Fully Connected Layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.Dropout(0.5),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, S * S * (C + B * 5)),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x.view(-1, self.S, self.S, self.C + self.B * 5)


In [6]:
model = YOLOv1(S=7, B=2, C=1)
x = torch.randn((2, 3, 448, 448))  # batch of 2 images
output = model(x)
print(output.shape)  # torch.Size([2, 7, 7, 11])


torch.Size([2, 7, 7, 11])


In [7]:
import torch
import torch.nn as nn

class YOLOLoss(nn.Module):
    def __init__(self, S=7, B=2, C=1, lambda_coord=5, lambda_noobj=0.5):
        super(YOLOLoss, self).__init__()
        self.mse = nn.MSELoss(reduction="sum")

        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def compute_iou(self, box1, box2):
        box1_x1 = box1[..., 0] - box1[..., 2] / 2
        box1_y1 = box1[..., 1] - box1[..., 3] / 2
        box1_x2 = box1[..., 0] + box1[..., 2] / 2
        box1_y2 = box1[..., 1] + box1[..., 3] / 2

        box2_x1 = box2[..., 0] - box2[..., 2] / 2
        box2_y1 = box2[..., 1] - box2[..., 3] / 2
        box2_x2 = box2[..., 0] + box2[..., 2] / 2
        box2_y2 = box2[..., 1] + box2[..., 3] / 2

        inter_x1 = torch.max(box1_x1, box2_x1)
        inter_y1 = torch.max(box1_y1, box2_y1)
        inter_x2 = torch.min(box1_x2, box2_x2)
        inter_y2 = torch.min(box1_y2, box2_y2)

        inter_area = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
        box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
        box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

        union_area = box1_area + box2_area - inter_area + 1e-6
        return inter_area / union_area

    def forward(self, predictions, target):
        N = predictions.shape[0]
        predictions = predictions.view(N, self.S, self.S, self.C + self.B * 5)

        # split predictions
        pred_classes = predictions[..., :self.C]
        pred_boxes = predictions[..., self.C:].view(N, self.S, self.S, self.B, 5)

        # split target
        target_classes = target[..., :self.C]
        target_boxes = target[..., self.C:].view(N, self.S, self.S, self.B, 5)

        # find IOU for the 2 boxes
        ious = self.compute_iou(pred_boxes[..., :4], target_boxes[..., :4])
        _, bestbox = ious.max(-1)

        exists_object = target[..., self.C].unsqueeze(-1)

        coord_loss = 0
        conf_loss = 0
        noobj_loss = 0
        class_loss = 0

        for b in range(self.B):
            box_pred = pred_boxes[..., b, :]
            box_target = target_boxes[..., b, :]

            # Identity the best box (only for that b)
            mask = bestbox == b
            mask = mask.unsqueeze(-1)

            pred_box = box_pred[..., :4]
            target_box = box_target[..., :4]

            box_confidence = box_pred[..., 4:5]
            target_confidence = box_target[..., 4:5]

            # Loss terms
            coord_loss += self.lambda_coord * self.mse(
                exists_object * mask * pred_box,
                exists_object * mask * target_box
            )

            conf_loss += self.mse(
                exists_object * mask * box_confidence,
                exists_object * mask * target_confidence
            )

            noobj_loss += self.lambda_noobj * self.mse(
                (1 - exists_object) * box_confidence,
                torch.zeros_like(box_confidence)
            )

        class_loss = self.mse(
            exists_object * pred_classes,
            exists_object * target_classes
        )

        total_loss = coord_loss + conf_loss + noobj_loss + class_loss
        return total_loss / N


In [8]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0

    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)

        preds = model(images)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            preds = model(images)
            loss = loss_fn(preds, targets)
            total_loss += loss.item()

    return total_loss / len(dataloader)


In [9]:
import torch.optim as optim

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YOLOv1(S=7, B=2, C=1).to(device)
loss_fn = YOLOLoss(S=7, B=2, C=1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Assuming train_loader and val_loader are already defined
num_epochs = 50

best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss = validate(model, val_loader, loss_fn, device)

    print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": best_val_loss
        }
        torch.save(checkpoint, "best_yolov1_model.pth")
        print(f"Saved new best model at epoch {epoch+1} with val loss {val_loss:.4f}")



[Epoch 1/50] Train Loss: 19.7882 | Val Loss: 19.4648
Saved new best model at epoch 1 with val loss 19.4648
[Epoch 2/50] Train Loss: 13.5123 | Val Loss: 14.9721
Saved new best model at epoch 2 with val loss 14.9721
[Epoch 3/50] Train Loss: 12.2757 | Val Loss: 9.9175
Saved new best model at epoch 3 with val loss 9.9175
[Epoch 4/50] Train Loss: 12.8926 | Val Loss: 230.4607
[Epoch 5/50] Train Loss: 11.9188 | Val Loss: 19.4346
[Epoch 6/50] Train Loss: 11.3203 | Val Loss: 10.2141
[Epoch 7/50] Train Loss: 15.5885 | Val Loss: 10.7273
[Epoch 8/50] Train Loss: 12.0568 | Val Loss: 14.3066
[Epoch 9/50] Train Loss: 11.0200 | Val Loss: 10.9231
[Epoch 10/50] Train Loss: 10.4827 | Val Loss: 8.3905
Saved new best model at epoch 10 with val loss 8.3905
[Epoch 11/50] Train Loss: 9.6766 | Val Loss: 8.5977
[Epoch 12/50] Train Loss: 9.4226 | Val Loss: 7.6796
Saved new best model at epoch 12 with val loss 7.6796
[Epoch 13/50] Train Loss: 8.5118 | Val Loss: 7.5077
Saved new best model at epoch 13 with val los

In [47]:
import torch
import torch.optim as optim

checkpoint = torch.load("best_yolov1_model.pth", map_location='cpu')

model = YOLOv1(S=7, B=2, C=1)
model.load_state_dict(checkpoint["model_state_dict"])

device = torch.device("cpu")
model = model.to(device)
model.eval()

optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print(f"Model loaded from epoch {checkpoint['epoch']}, val loss: {checkpoint['val_loss']:.4f}")


Model loaded from epoch 49, val loss: 5.2905


In [48]:
from PIL import Image
import torchvision.transforms as transforms

# Resize and normalize input image
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    return image, input_tensor.to(device)


In [49]:
def cellboxes_to_boxes(predictions, S=7, B=2, C=1, conf_threshold=0.1):
    predictions = predictions.squeeze(0)  # Shape: [7, 7, 11]
    boxes = []

    for i in range(S):
        for j in range(S):
            cell = predictions[i, j]

            # Class label and confidence
            class_score = cell[0]
            box1 = cell[1:5]
            conf1 = cell[5]
            box2 = cell[6:10]
            conf2 = cell[10]

            # Choose box with higher confidence
            if conf1 * class_score > conf2 * class_score:
                box = box1
                conf = conf1 * class_score
            else:
                box = box2
                conf = conf2 * class_score

            if conf < conf_threshold:
                continue

            # Convert box coordinates from cell-relative to image-relative
            x_rel, y_rel, w_rel, h_rel = box
            x = (j + x_rel) / S
            y = (i + y_rel) / S
            w = w_rel / S
            h = h_rel / S

            x1 = x - w / 2
            y1 = y - h / 2
            x2 = x + w / 2
            y2 = y + h / 2

            boxes.append([x1, y1, x2, y2, conf])

    return boxes


In [50]:
from PIL import ImageDraw

def draw_boxes(image, boxes, color="red"):
    draw = ImageDraw.Draw(image)
    w, h = image.size

    for box in boxes:
        x1, y1, x2, y2, conf = box
        draw.rectangle([x1 * w, y1 * h, x2 * w, y2 * h], outline=color, width=2)
        draw.text((x1 * w, y1 * h), f"{conf:.2f}", fill=color)

    image.show()


In [51]:
def detect_faces(image_path, model, device, conf_threshold=0.2):
    image, input_tensor = preprocess_image(image_path)

    with torch.no_grad():
        predictions = model(input_tensor)
        boxes = cellboxes_to_boxes(predictions, conf_threshold=conf_threshold)

    draw_boxes(image, boxes)
    print(f"Detected {len(boxes)} face(s) with conf > {conf_threshold}")


In [58]:
detect_faces("WIDERFACE/images/wider_0.jpg", model, device, conf_threshold=0.02)


Detected 0 face(s) with conf > 0.02


I0000 00:00:1751459418.231209   61676 voice_transcription.cc:58] Registering VoiceTranscriptionCapability
Created TensorFlow Lite XNNPACK delegate for CPU.
Attempting to use a delegate that only supports static-sized tensors with a graph that has dynamic-sized tensors (tensor#-1 is a dynamic-sized tensor).


In [59]:
detect_faces("Input_images/test.jpg", model, device, conf_threshold=0.001)

Detected 5 face(s) with conf > 0.001


I0000 00:00:1751459448.586924   61855 voice_transcription.cc:58] Registering VoiceTranscriptionCapability
Created TensorFlow Lite XNNPACK delegate for CPU.
Attempting to use a delegate that only supports static-sized tensors with a graph that has dynamic-sized tensors (tensor#-1 is a dynamic-sized tensor).
